In [ ]:
import json, os, copy
from pathlib import Path
import numpy as np
from scipy.special import logsumexp, softmax
from threadpoolctl import threadpool_limits
threadpool_limits(limits=1)

In [ ]:
def carry(input_dir, output_dir):
    # preserve the dependency chain without duplicating upstream data
    for source in input_dir.iterdir():
        target = output_dir / source.name
        target.unlink(missing_ok=True)
        target.symlink_to(Path('../input') / source.name)

def clone(net):
    return {k: v.copy() for k, v in net.items()}

def save_nets(path, nets):
    np.savez_compressed(path, **{f'{name}__{key}': value for name, net in nets.items() for key, value in net.items()})

def load_nets(path):
    with np.load(path) as data:
        names = sorted({key.split('__')[0] for key in data.files})
        return {name: {key.split('__')[1]: data[key].copy() for key in data.files if key.startswith(name + '__')} for name in names}

In [ ]:
def initialize(rng, width, blocks):
    return {'stem_w': rng.normal(0, 1 / 8, (64, width)), 'stem_b': np.zeros(width),
            'block_w': rng.normal(0, 1 / np.sqrt(width), (blocks, width, width)),
            'block_b': np.zeros((blocks, width)),
            'head_w': rng.normal(0, 1 / np.sqrt(width), (width, 10)), 'head_b': np.zeros(10)}

def forward(net, x, cache=False):
    h = np.tanh(x @ net['stem_w'] + net['stem_b'])
    hidden, changes = [h], []
    alpha = 1 / np.sqrt(len(net['block_w']))
    for w, b in zip(net['block_w'], net['block_b']):
        t = np.tanh(h @ w + b)
        h = h + alpha * t
        hidden.append(h)
        changes.append(t)
    logits = h @ net['head_w'] + net['head_b']
    return (logits, hidden, changes) if cache else logits

def loss_grad(net, x, y):
    logits, hidden, changes = forward(net, x, cache=True)
    loss = np.mean(logsumexp(logits, axis=1) - logits[np.arange(len(y)), y])
    dl = softmax(logits, axis=1)
    dl[np.arange(len(y)), y] -= 1
    dl /= len(y)
    grad = {k: np.zeros_like(v) for k, v in net.items()}
    grad['head_w'], grad['head_b'] = hidden[-1].T @ dl, dl.sum(axis=0)
    dh = dl @ net['head_w'].T
    alpha = 1 / np.sqrt(len(changes))
    for j in range(len(changes) - 1, -1, -1):
        dz = alpha * dh * (1 - changes[j] ** 2)
        grad['block_w'][j], grad['block_b'][j] = hidden[j].T @ dz, dz.sum(axis=0)
        dh = dh + dz @ net['block_w'][j].T
    dz = dh * (1 - hidden[0] ** 2)
    grad['stem_w'], grad['stem_b'] = x.T @ dz, dz.sum(axis=0)
    return loss, grad

In [ ]:
def train(net, x, y, steps, rng, lr, batch, probabilities=None, optimizer=None):
    # retain optimizer state across saved fine-tuning checkpoints
    if optimizer is None:
        optimizer = {'t': 0, 'm': {k: np.zeros_like(v) for k, v in net.items()},
                     'v': {k: np.zeros_like(v) for k, v in net.items()}}
    for _ in range(steps):
        ix = rng.choice(len(y), batch, replace=True, p=probabilities)
        _, grad = loss_grad(net, x[ix], y[ix])
        optimizer['t'] += 1
        t = optimizer['t']
        for key in net:
            g = grad[key] + 1e-4 * net[key]
            optimizer['m'][key] = 0.9 * optimizer['m'][key] + 0.1 * g
            optimizer['v'][key] = 0.999 * optimizer['v'][key] + 0.001 * g ** 2
            m = optimizer['m'][key] / (1 - 0.9 ** t)
            v = optimizer['v'][key] / (1 - 0.999 ** t)
            net[key] -= lr * m / (np.sqrt(v) + 1e-8)
    return optimizer

def digit_metrics(net, x, y):
    logits = forward(net, x)
    return {'digit_accuracy': float(np.mean(logits.argmax(axis=1) == y)),
            'digit_nll': float(np.mean(logsumexp(logits, axis=1) - logits[np.arange(len(y)), y]))}

def block_vectors(net):
    return np.concatenate([net['block_w'].reshape(len(net['block_w']), -1), net['block_b']], axis=1)

In [ ]:
def ancestry_path(rng, blocks, sources, mean_tract, kind='mosaic'):
    path = np.empty(blocks, dtype=int)
    path[0] = rng.integers(sources)
    for j in range(1, blocks):
        choices = np.delete(np.arange(sources), path[j - 1])
        path[j] = rng.choice(choices) if rng.random() < 1 / mean_tract else path[j - 1]
    return np.full(blocks, path[0]) if kind == 'single' else path

def splice(root, donors, path):
    net = clone(root)
    for j, source in enumerate(path):
        net['block_w'][j] = donors[source]['block_w'][j]
        net['block_b'][j] = donors[source]['block_b'][j]
    return net

def permute(net, p):
    # one shared hidden permutation preserves the entire residual net function
    out = clone(net)
    out['stem_w'], out['stem_b'] = net['stem_w'][:, p], net['stem_b'][p]
    out['block_w'], out['block_b'] = net['block_w'][:, p][:, :, p], net['block_b'][:, p]
    out['head_w'] = net['head_w'][p]
    return out

In [ ]:
def copy_cost(query, references, indices, scale):
    # each reference checkpoint is a copying state, as in a haplotype panel
    r = references.reshape(-1, references.shape[-2], references.shape[-1])
    distances = np.mean((r[:, :, indices] - query[None, :, indices]) ** 2, axis=-1) / scale[None]
    return distances.T

def source_probabilities(probabilities, sources=3):
    # aggregate checkpoint states to their source lineage
    return probabilities.reshape(*probabilities.shape[:-1], sources, -1).sum(axis=-1)

def viterbi(costs, lam):
    costs = np.asarray(costs)
    one = costs.ndim == 2
    costs = costs[None] if one else costs
    n, length, k = costs.shape
    penalty = lam * (1 - np.eye(k))
    scores = costs[:, 0].copy()
    back = np.zeros((n, length, k), dtype=int)
    for j in range(1, length):
        candidate = scores[:, :, None] + penalty[None]
        back[:, j] = candidate.argmin(axis=1)
        scores = costs[:, j] + candidate.min(axis=1)
    paths = np.zeros((n, length), dtype=int)
    paths[:, -1] = scores.argmin(axis=1)
    for j in range(length - 1, 0, -1):
        paths[:, j - 1] = back[np.arange(n), j, paths[:, j]]
    return paths[0] if one else paths

def posterior(costs, lam, temperature):
    costs = np.asarray(costs)
    one = costs.ndim == 2
    costs = costs[None] if one else costs
    n, length, k = costs.shape
    emission = -costs / temperature
    transition = -lam * (1 - np.eye(k)) / temperature
    transition -= logsumexp(transition, axis=1, keepdims=True)
    alpha, beta = np.empty_like(costs), np.zeros_like(costs)
    alpha[:, 0] = emission[:, 0] - np.log(k)
    for j in range(1, length):
        alpha[:, j] = emission[:, j] + logsumexp(alpha[:, j - 1, :, None] + transition[None], axis=1)
    for j in range(length - 2, -1, -1):
        beta[:, j] = logsumexp(transition[None] + emission[:, j + 1, None] + beta[:, j + 1, None], axis=2)
    logp = alpha + beta
    p = np.exp(logp - logsumexp(logp, axis=2, keepdims=True))
    return p[0] if one else p

In [ ]:
def boundary_counts(truth, prediction, tolerance=0):
    a = list(np.flatnonzero(truth[1:] != truth[:-1]) + 1)
    b = list(np.flatnonzero(prediction[1:] != prediction[:-1]) + 1)
    # maximum one-to-one matching on ordered boundary lists
    i = j = matched = 0
    while i < len(a) and j < len(b):
        if abs(a[i] - b[j]) <= tolerance:
            matched += 1
            i += 1
            j += 1
        elif a[i] < b[j]:
            i += 1
        else:
            j += 1
    return matched, len(b) - matched, len(a) - matched

def score_path(truth, prediction, probabilities):
    tp, fp, fn = boundary_counts(truth, prediction)
    t1, f1, n1 = boundary_counts(truth, prediction, tolerance=1)
    known = truth < probabilities.shape[1]
    targets = np.eye(probabilities.shape[1])[truth[known]]
    p = probabilities[known]
    return {'block_accuracy': float(np.mean(truth == prediction)),
            'boundary_tp': tp, 'boundary_fp': fp, 'boundary_fn': fn,
            'boundary_f1': 2 * tp / (2 * tp + fp + fn) if tp + fp + fn else 1.0,
            'boundary_f1_tolerant': 2 * t1 / (2 * t1 + f1 + n1) if t1 + f1 + n1 else 1.0,
            'known_nll': float(-np.log(np.maximum(p[np.arange(len(p)), truth[known]], 1e-300)).mean()),
            'known_brier': float(np.mean(np.sum((p - targets) ** 2, axis=1))),
            'predicted_switches': int(np.sum(prediction[1:] != prediction[:-1])),
            'true_switches': int(np.sum(truth[1:] != truth[:-1]))}